In [1]:
import numpy as np
import pandas as pd
import pickle
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np 
from collections import Counter
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [2]:
# 1. Emotion Mapping Function
def map_videos_to_labels():
    emotion_map = {
        1: 0, 2: 0, 3: 0,  # Anger
        4: 1, 5: 1, 6: 1,  # Disgust
        7: 2, 8: 2, 9: 2,  # Fear
        10: 3, 11: 3, 12: 3,  # Sadness
        17: 4, 18: 4, 19: 4,  # Amusement
        20: 5, 21: 5, 22: 5,  # Inspiration
        23: 6, 24: 6, 25: 6,  # Joy
        26: 7, 27: 7, 28: 7,  # Tenderness
        13: 8, 14: 8, 15: 8, 16: 8  # Neutral
    }
    return emotion_map

In [3]:
# 2. Naming Emotion
def create_emotions_from_map():
    emotion_map = map_videos_to_labels()  # Get the emotion mapping
    emotions = [''] * 28  # Initialize an empty list with 28 categories
    
    # Create emotion names based on the mapping
    for key, value in emotion_map.items():
        if value == 0:
            emotions[key-1] = 'Anger'
        elif value == 1:
            emotions[key-1] = 'Disgust'
        elif value == 2:
            emotions[key-1] = 'Fear'
        elif value == 3:
            emotions[key-1] = 'Sadness'
        elif value == 4:
            emotions[key-1] = 'Amusement'
        elif value == 5:
            emotions[key-1] = 'Inspiration'
        elif value == 6:
            emotions[key-1] = 'Joy'
        elif value == 7:
            emotions[key-1] = 'Tenderness'
        elif value == 8:
            emotions[key-1] = 'Neutral'

    return emotions

In [4]:
def load_features(data_path):
    """
    Load DE and PSD features for all subjects, and map them to labels per second.
    """
    all_data = []  # Store data for all subjects
    emotion_map = map_videos_to_labels()  # Get the mapping of video IDs to emotions

    for subject in range(123):
        # File paths
        de_file = os.path.join(data_path, 'DE', f'sub{subject:03d}.pkl.pkl')
        psd_file = os.path.join(data_path, 'PSD', f'sub{subject:03d}.pkl.pkl')

        # Check file existence
        if not os.path.exists(de_file) or not os.path.exists(psd_file):
            print(f"File missing for subject {subject}: {de_file} or {psd_file}")
            continue

        # Load DE and PSD features
        try:
            with open(de_file, 'rb') as f:
                de_features = pickle.load(f)
            with open(psd_file, 'rb') as f:
                psd_features = pickle.load(f)
        except Exception as e:
            print(f"Error loading files for subject {subject}: {e}")
            continue

        # Ensure DE and PSD features are correctly arranged as 4D arrays
        de_data_4d = np.expand_dims(de_features, axis=0) if de_features.ndim <= 3 else de_features
        psd_data_4d = np.expand_dims(psd_features, axis=0) if psd_features.ndim <= 3 else psd_features

        # Retrieve dimensions
        video_num, elec_num, trial_dur, freq_band = de_data_4d.shape
        
        # Combined results will have shape (VideoNum * TrialDur, 321)
        result_2d = np.zeros((video_num * trial_dur, 321))

        for v in range(video_num):
            for t in range(trial_dur):
                # Flatten the DE and PSD features into a single array
                de_flattened_values = de_data_4d[v, :, t, :].flatten()     # Shape: (160,)
                psd_flattened_values = psd_data_4d[v, :, t, :].flatten()   # Shape: (160,)
                
                # Combine DE and PSD features
                combined_values = np.concatenate((de_flattened_values, psd_flattened_values))

                # Get the emotion label for the current VideoNum (index + 1)
                emotion_label = emotion_map.get(v + 1, -1)

                # Fill the result array
                result_2d[v * trial_dur + t, :320] = combined_values  # First 320 columns
                result_2d[v * trial_dur + t, 320] = int(emotion_label)  # Last column for the label

        # Append this subject's 2D data to all_data
        all_data.append(result_2d)

    # Combine all subjects' data into a single 2D array
    if all_data:
        final_data = np.vstack(all_data)
    else:
        final_data = np.array([])  # Handle the case if no data was loaded
    
    return final_data

# Example usage:
# data_path = 'path_to_your_data_directory'
# features_and_labels = load_features(data_path)

In [5]:
def evaluate_classifiers(X_train, X_test, y_train, y_test, emotions):
    """
    Compare multiple classifiers.
    """
    classifiers = {
        # 'Support Vector Machine': SVC(kernel='rbf'),
        # 'Random Forest': RandomForestClassifier(n_estimators=500),
        # 'Neural Network': MLPClassifier(max_iter=1000),
        # 'K-Nearest Neighbors': KNeighborsClassifier(),
        'Decision Tree': DecisionTreeClassifier()
    }

    results = {}

    for name, clf in classifiers.items():
        # Train and evaluate
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        # Store results
        results[name] = {
            'classification_report': classification_report(y_test, y_pred, target_names = emotions),
            'confusion_matrix': confusion_matrix(y_test, y_pred)
        }

    return results

In [6]:
def run_study(data_path):
    # Load features
    X = load_features(data_path)
    y = X[:, -1]

    # Class distribution
    class_counts = Counter(y)
    print("Class distribution:", class_counts)

    # Feature scaling
    scaler = StandardScaler()
    X_combined = X[:, :-1]
    X_combined = scaler.fit_transform(X_combined)

    # Prepare data
    X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.15, random_state=42)

    emotions = [
        'Anger', 'Disgust', 'Fear', 'Sadness',
        'Amusement', 'Inspiration', 'Joy', 'Tenderness', 'Neutral'
    ]
    print("Emotion categories:", emotions)

    # 7. Evaluate classifiers
    results = evaluate_classifiers(X_train, X_test, y_train, y_test, emotions)

    return results


In [7]:
# 6. Example usage to test the run_study function
if __name__ == '__main__':
    study_results = run_study('')  # Replace with your actual data path

    # Print results for each classifier
    for clf_name, clf_results in study_results.items():
        print(f"\n{clf_name} Results:")
        print(clf_results['classification_report'])

Class distribution: Counter({np.float64(8.0): 14760, np.float64(0.0): 11070, np.float64(1.0): 11070, np.float64(2.0): 11070, np.float64(3.0): 11070, np.float64(4.0): 11070, np.float64(5.0): 11070, np.float64(6.0): 11070, np.float64(7.0): 11070})
Emotion categories: ['Anger', 'Disgust', 'Fear', 'Sadness', 'Amusement', 'Inspiration', 'Joy', 'Tenderness', 'Neutral']

Decision Tree Results:
              precision    recall  f1-score   support

       Anger       0.29      0.29      0.29      1652
     Disgust       0.32      0.33      0.32      1630
        Fear       0.33      0.32      0.32      1740
     Sadness       0.28      0.27      0.28      1674
   Amusement       0.30      0.30      0.30      1711
 Inspiration       0.26      0.27      0.27      1645
         Joy       0.25      0.26      0.25      1679
  Tenderness       0.30      0.29      0.29      1624
     Neutral       0.37      0.37      0.37      2143

    accuracy                           0.30     15498
   macro avg  